In [3]:
import json
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import sqlite3
from time import time

measurement_df[measurement_df["EXFOR_Entry"]=="22764"]

entry_df["available"] = "unknown"
availability_unc = {
    "available": ["13632", "12988", "22052", "10725", "13587", 
                  "12972", "13840", "13907", "13588", "30077", 
                  "20638", "10982","10663", "12751", "20066", 
                  "10193", "21193", "12830","11679", "30390", 
                  "12853", "30502", "30234", "22451","13753",
                  "20789", "40061", "30248", "22580", "22838"],
    "paywalled": ["10724", "14349", "12941", "23534", "23605"],
    "conference": ["22314", "13730", "22331", "12897", "12971", "13734", "10955", "13511",
                  "21325", "20939"],
    "private": ["22333", "13767", "22332", "22315", "41225", "14261", "13768", "22316"],
    "lab_report": ["14249", "12816", "11010", "10618"],
    "data_only": ["22360", "20435", "21270", "20300", "10366", "21146",
                 "20718", "22360"],
    "french": ["20118", "20445"],
    "russian": ["40729"]

}

availability_no_unc = {
    "available": ["22764", "12278", "12886", "20115", "11913",
                 "11601"]
}
    
display(entry_df[entry_df["Uncertainty_Complete"]==1].sort_values("Num_Subentries", ascending=False)[30:45])

#display(entry_df[entry_df["Uncertainty_Complete"]==1].sort_values("Num_Measurements", ascending=False)[30:])

#display(entry_df[entry_df["Uncertainty_Complete"]==0].sort_values("Num_Measurements", ascending=False)[:30])

In [4]:
conn = sqlite3.connect('output/nugrade_data.db')
entry_df = pd.read_sql('SELECT * FROM entries', conn)
report_df = pd.read_sql('SELECT * FROM report_embeddings', conn)
report_df['mean_embedding'] = report_df['mean_embedding'].apply(lambda x: np.frombuffer(x, dtype=np.float32))
measurement_df = pd.read_sql('SELECT * FROM measurements', conn)
measurement_df = measurement_df[measurement_df["Energy"] > 0]
measurement_df["dData_adopted"] = measurement_df["dData_assumed"]
measurement_df["uncertainty_source"] = "N/A"
measurement_df.loc[~measurement_df["dData"].isna(), "uncertainty_source"] = "included"
subentry_df = pd.read_sql('SELECT * FROM subentries', conn)
conn.close()

## Standardization of structural values

In [14]:
MEAN_LOG_ENERGY = np.log10(measurement_df["Energy"]).mean()
STD_LOG_ENERGY = np.log10(measurement_df["Energy"]).std()
measurement_df["Energy_Logstd"] = (np.log10(measurement_df["Energy"])-MEAN_LOG_ENERGY)/STD_LOG_ENERGY

MEAN_Z = measurement_df["Z"].mean()
STD_Z = measurement_df["Z"].std()
measurement_df["Z_std"] = (measurement_df["Z"]-MEAN_Z)/STD_Z

MEAN_A = measurement_df["A"].mean()
STD_A = measurement_df["A"].std()
measurement_df["A_std"] = (measurement_df["A"]-MEAN_A)/STD_A

## Finding Missing Uncertainites

In [19]:
missing_indices = measurement_df[measurement_df["dData"].isna()]
#sample_impule_indices = measurement_df[measurement_df["dData"].isna()].sample(100, random_state=42).index
report_entries = set(report_df["EXFOR_Entry"].values)
reports_with_uncertainty = report_df[report_df["EXFOR_Entry"].isin(entry_df[entry_df["Uncertainty_Complete"] == 1]["EXFOR_Entry"])]
reports_without_uncertainty = report_df[report_df["EXFOR_Entry"].isin(entry_df[entry_df["Uncertainty_Complete"] == 0]["EXFOR_Entry"])]


sample_impule_indices = measurement_df[measurement_df["EXFOR_Entry"].isin(reports_without_uncertainty["EXFOR_Entry"])].index

In [20]:
missing_measurements = len(missing_indices)
total_measurements = len(measurement_df)
number_of_entries = len(measurement_df['EXFOR_Entry'].unique())
number_of_subentries = len(measurement_df['EXFOR_Subentry'].unique())
number_of_reports_parsed = len(report_df)

print(f"Total measurements: {total_measurements}")
print(f"Missing measurements: {missing_measurements} ({round(100*missing_measurements/total_measurements,1)}%)")
print(f"Total EXFOR entries: {number_of_entries}")
print(f"Number of EXFOR entries with embeddings: {number_of_reports_parsed} ({len(reports_with_uncertainty)} with uncertainty)")
print(f"Total EXFOR subentries: {number_of_subentries}")

Total measurements: 2611611
Missing measurements: 294637 (11.3%)
Total EXFOR entries: 2193
Number of EXFOR entries with embeddings: 39 (33 with uncertainty)
Total EXFOR subentries: 6193


In [21]:
measurement_df[measurement_df["EXFOR_Entry"].isin(reports_without_uncertainty["EXFOR_Entry"])]

,Reaction,Projectile,Energy,dEnergy,Data,dData,MT,Dataset_Number,EXFOR_Entry,EXFOR_Subentry,...,endf7-1_chi_squared,endf7-1_relative_error,endf8,endf8_chi_squared,endf8_relative_error,dData_adopted,Energy_Logstd,Z_std,A_std,uncertainty_source
184352,"N,TOT",n,9593.1,NaN,2.4698,1.25160,1,12886002,12886,12886002,...,0.395993,39.869122,1.765794,0.395993,39.869122,1.25160,-0.049503,-1.211006,-1.180983,included
184353,"N,TOT",n,9603.8,NaN,3.9837,1.25340,1,12886002,12886,12886002,...,3.925245,125.626804,1.765615,3.925245,125.626804,1.25340,-0.049259,-1.211006,-1.180983,included
184354,"N,TOT",n,9614.4,NaN,3.0292,1.20880,1,12886002,12886,12886002,...,1.321224,71.583534,1.765437,1.321224,71.583534,1.20880,-0.049018,-1.211006,-1.180983,included
184355,"N,TOT",n,9625.1,NaN,0.6790,1.24470,1,12886002,12886,12886002,...,0.947986,-61.535380,1.765259,0.947986,-61.535380,1.24470,-0.048775,-1.211006,-1.180983,included
184356,"N,TOT",n,9635.8,NaN,3.7187,1.20100,1,12886002,12886,12886002,...,3.177879,110.681708,1.765080,3.177879,110.681708,1.20100,-0.048532,-1.211006,-1.180983,included
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1214476,"N,TOT",n,119750.0,59.875,5.8248,0.14060,1,10639005,10639,10639005,...,1.784531,-7.918554,6.279781,1.472320,-7.245180,0.14060,0.502030,-0.147596,-0.167681,included
1214477,"N,TOT",n,119840.0,59.920,5.4357,0.13936,1,10639005,10639,10639005,...,5.683238,-14.068949,6.279813,5.112848,-13.441688,0.13936,0.502194,-0.147596,-0.167681,included
1214478,"N,TOT",n,119930.0,59.965,5.6199,0.13795,1,10639005,10639,10639005,...,3.610109,-11.156274,6.279844,3.157132,-10.508927,0.13795,0.502358,-0.147596,-0.167681,included
1214479,"N,TOT",n,120010.0,60.005,5.3045,0.13663,1,10639005,10639,10639005,...,7.630497,-16.141756,6.279872,6.962971,-15.531720,0.13663,0.502504,-0.147596,-0.167681,included


In [22]:
similarity_score_labels = ['background_treatment_max_sim',
       'detector_efficiency_max_sim', 'normalization_treatment_max_sim',
       'sample_attenuation_scattering_corrections_max_sim',
       'dead_time_max_sim', 'coincidence_max_sim',
       'statistical_uncertainty_max_sim', 'uncertainty_propagation_max_sim',
       'time_of_flight_max_sim']

inner_z_weight = 1.5
inner_a_weight = 1.0
inner_energy_weight = 0.5

weights = pd.Series({'background_treatment_max_sim':1, 
                     'detector_efficiency_max_sim':1,
                     'normalization_treatment_max_sim':1,
                     'sample_attenuation_scattering_corrections_max_sim':1,
                     'dead_time_max_sim':1, 
                     'coincidence_max_sim':1,
                     'statistical_uncertainty_max_sim':1,
                     'uncertainty_propagation_max_sim':1,
                     'time_of_flight_max_sim':1, 
                     'Energy_Logstd':1, 
                     'Z_std':1, 
                     'A_std':1,
                     'mean_embedding':1})

k_neighbors = 5

In [24]:
grouped = {
    (entry, mt): group 
    for (entry, mt), group in measurement_df.groupby(["EXFOR_Entry", "MT"])
}
num_processed = 0 
for index in sample_impule_indices:
    impute_data = measurement_df.iloc[index]
    impute_exfor_entry = impute_data["EXFOR_Entry"]
    
    if impute_exfor_entry not in report_entries:
        print("Falling back to manual impute")
        measurement_df.loc[index, "uncertainty_source"] = "quantile_imputed"
        measurement_df.loc[index,"dData_adopted"] = measurement_df["dData_assumed"].iloc[index]    
        continue
        
    impute_embeddings = report_df[report_df["EXFOR_Entry"]==impute_exfor_entry].drop(columns="EXFOR_Entry").iloc[0]
    impute_features = pd.concat([impute_data, impute_embeddings])
    impute_Z = measurement_df.iloc[index]["Z"]
    knn_samples = []
    for report_exfor_entry in reports_with_uncertainty["EXFOR_Entry"]:
        candidates = grouped.get((report_exfor_entry, impute_data["MT"]))
        if candidates is None:
            continue
        squared_distances = inner_energy_weight*(candidates["Energy_Logstd"]-impute_data["Energy_Logstd"])**2 + \
            inner_z_weight*(candidates["Z_std"]-impute_data["Z_std"])**2 + \
            inner_a_weight*(candidates["A_std"]-impute_data["A_std"])**2
        closest_point = candidates.loc[squared_distances.idxmin()]
        report_info = report_df[report_df["EXFOR_Entry"]==report_exfor_entry].iloc[0]
        closest_point = pd.concat([closest_point, report_info])
        knn_samples.append(closest_point)
    knn_samples = pd.concat(knn_samples, axis=1).T
    differences = (impute_features[similarity_score_labels] - knn_samples[similarity_score_labels])**2
    differences["Energy_Logstd"] = (impute_features["Energy_Logstd"]-knn_samples["Energy_Logstd"])**2
    differences["Z_std"] = (impute_features["Z_std"]-knn_samples["Z_std"])**2
    differences["A_std"] = (impute_features["A_std"]-knn_samples["A_std"])**2
    
    mean_embedding_differences = 1 - cosine_similarity(impute_features["mean_embedding"].reshape(1,-1), 
                                           np.vstack(knn_samples["mean_embedding"].values) )
    differences["mean_embedding"] = mean_embedding_differences[0]
    differences = differences.apply(pd.to_numeric)
    distances = np.sqrt(weights*differences).sum(axis=1)
    neighbors = pd.DataFrame({"distance":distances, 
                              "rel_unc": knn_samples["dData"]/knn_samples["Data"]}
                            ).sort_values(by="distance")[:k_neighbors]
    
    neighbor_weights = 1 / neighbors["distance"]
    weighted_mean = np.average(neighbors["rel_unc"], weights=neighbor_weights)
    measurement_df.loc[index,"dData_adopted"] = weighted_mean*measurement_df.loc[192948,"Data"]
    measurement_df.loc[index, "uncertainty_source"] = "nlp_imputed"
    num_processed += 1
    if num_processed%1000 == 0:
        print(num_processed)

1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000


In [33]:
measurement_df.loc[measurement_df["uncertainty_source"] == "N/A", "uncertainty_source"] = "quantile_imputed"
measurement_df.drop(columns=["Energy_Logstd","Z_std","A_std"], errors="ignore", inplace=True)

In [30]:
conn = sqlite3.connect('output/nugrade_data.db')

In [34]:
measurement_df.to_sql('measurements', conn, if_exists='replace', index=True)

2611611

In [35]:
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_reaction_channel
    ON measurements (Z, A, MT, Projectile)
""")

conn.close()

In [36]:
conn = sqlite3.connect('output/test_data.db')

In [39]:
test_db = measurement_df[ (measurement_df["Z"] == 3) & (measurement_df["A"] == 7) & (measurement_df["MT"] == 1) & (measurement_df["Projectile"] == "n")]

In [41]:
test_db.to_sql('measurements', conn, if_exists='replace', index=True)
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_reaction_channel
    ON measurements (Z, A, MT, Projectile)
""")
conn.close()